# Quantization from Scratch: Scale, Zero-Point, GPTQ and AWQ

The runnable code of [Quantization from Scratch: Scale, Zero-Point, GPTQ and AWQ](https://residual-stream.lvtan71.workers.dev/posts/quantization-from-scratch/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/quantization-from-scratch.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 3 min end to end.
Downloads Qwen2.5-0.5B (about 1 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "quantization-from-scratch"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['data/blog_train.txt']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Measured on two real layers

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/quantization-from-scratch/#measured-on-two-real-layers)

**Not shown in the post:** the post's code reads real inputs of two layers of Qwen2.5-0.5B, captured beforehand. This cell recaptures them from the blog's text.

In [ ]:
import numpy as np
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

# Real inputs for the experiments: run Qwen2.5-0.5B (base) on the CPU over the
# blog's text and record what goes into the MLP down-projection (4,864 -> 896)
# of layers 12 and 22, with its weight. X_L: 4,096 tokens from the start of
# the text (2,048 to calibrate, 2,048 to measure). H_big_L, mag_big_L: 32,768
# tokens from later in the text, kept only as the sums GPTQ and AWQ need,
# H = 2 X^T X and the mean |x| per input channel.
PATH = snapshot_download("Qwen/Qwen2.5-0.5B", revision="060db6499f32faf8b98477b0a26969ef7d8b9987")
tok = AutoTokenizer.from_pretrained(PATH)
m = AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float32).eval()
text = open("blog_train.txt").read()
head = tok(text[:60_000], return_tensors="pt").input_ids[0]
tail = tok(text[60_000:], return_tensors="pt").input_ids[0]
for L in (12, 22):
    layer = m.model.layers[L].mlp.down_proj
    seen, H, absum, n = [], np.zeros((4864, 4864)), np.zeros(4864), 0
    hook = layer.register_forward_hook(lambda mod, inp, out: seen.append(inp[0][0].detach()))
    with torch.no_grad():
        for s in range(0, 4096, 512):
            m(head[s:s + 512][None])
    hook.remove()
    np.save(f"X_{L}.npy", torch.cat(seen).numpy()); np.save(f"W_{L}.npy", layer.weight.detach().numpy())
    def big(mod, inp, out):
        global H, absum, n
        x = inp[0][0].double().numpy()
        H += 2 * x.T @ x; absum += np.abs(x).sum(0); n += len(x)
    hook = layer.register_forward_hook(big)
    with torch.no_grad():
        for s in range(0, 32768, 512):
            m(tail[s:s + 512][None])
    hook.remove()
    np.save(f"H_big_{L}.npy", H); np.save(f"mag_big_{L}.npy", absum / n)
    print(f"layer {L}: inputs {torch.cat(seen).shape[0]} x 4864, calibration tokens {n}")

In [ ]:
import numpy as np

# Quantizing real layers: Qwen2.5-0.5B's MLP down-projections (896 x 4,864)
# in layers 12 and 22, with their real inputs from this blog's text. Quality is the
# layer's output error on 2,048 held-out tokens, relative to the output's
# size: ||X W^T - X Q^T|| / ||X W^T||. Calibration (GPTQ, AWQ) uses
# 32,768 other tokens, kept as H = 2 X^T X and each channel's mean |x|
# (capture_big.py), or, to show what too little does, 2,048.
def quantize(w, bits, axis=None):
    """Asymmetric round-to-nearest: an integer grid 0 .. 2^bits - 1, a scale
    and a zero-point so the grid spans [min, max] along `axis`."""
    lo, hi = w.min(axis=axis, keepdims=True), w.max(axis=axis, keepdims=True)
    scale = (hi - lo) / (2 ** bits - 1)
    scale = np.where(scale == 0, 1, scale)
    zero = np.round(-lo / scale)
    q = np.clip(np.round(w / scale) + zero, 0, 2 ** bits - 1)
    return (q - zero) * scale

def symmetric(w, bits, axis):
    """Symmetric (absmax): a grid centred on zero, -(2^(b-1) - 1) .. 2^(b-1) - 1,
    scale = max|w| / that; no zero-point to store."""
    top = 2 ** (bits - 1) - 1
    scale = np.abs(w).max(axis=axis, keepdims=True) / top
    scale = np.where(scale == 0, 1, scale)
    return np.clip(np.round(w / scale), -top, top) * scale

# NF4 (Dettmers et al., 2023): 16 levels at quantiles of a normal
# distribution, scaled per block by the block's absmax (QLoRA's values).
NF4 = np.array([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
                -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
                0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
                0.7229568362236023, 1.0])
def nf4(w, axis):
    m = np.abs(w).max(axis=axis, keepdims=True)
    m = np.where(m == 0, 1, m)
    x = w / m
    return NF4[np.abs(x[..., None] - NF4).argmin(-1)] * m

def rtn(W, bits, granularity):
    if granularity == "per tensor":
        return quantize(W, bits)
    if granularity == "per channel":
        return quantize(W, bits, axis=1)                      # one scale per output row
    G = W.reshape(W.shape[0], -1, 128)                        # groups of 128 inputs per row
    return quantize(G, bits, axis=2).reshape(W.shape)

def gptq(W, bits, H, block=128, damp=0.01):
    """GPTQ (Frantar et al., 2023): quantize one input column at a time, in
    groups of 128, and push each column's rounding error onto the columns
    not yet quantized, weighted by the inverse Hessian of the layer's
    squared output error, H = 2 X^T X."""
    W = W.copy(); n = W.shape[1]
    H = H.copy()
    H += damp * np.mean(np.diag(H)) * np.eye(n)
    Hinv = np.linalg.cholesky(np.linalg.inv(H)).T              # upper Cholesky factor of H^-1
    Q = np.zeros_like(W)
    for i1 in range(0, n, block):
        i2 = i1 + block
        W1, E1 = W[:, i1:i2].copy(), np.zeros((W.shape[0], block))
        lo, hi = W1.min(1), W1.max(1)                             # this group's grid, from current weights
        scale = np.where(hi > lo, (hi - lo) / (2 ** bits - 1), 1); zero = np.round(-lo / scale)
        for i in range(block):
            col = W1[:, i]
            q = (np.clip(np.round(col / scale) + zero, 0, 2 ** bits - 1) - zero) * scale
            Q[:, i1 + i] = q
            e = (col - q) / Hinv[i1 + i, i1 + i]
            W1[:, i:] -= np.outer(e, Hinv[i1 + i, i1 + i:i2])     # compensate within the block
            E1[:, i] = e
        W[:, i2:] -= E1 @ Hinv[i1:i2, i2:]                        # and in the columns after it
    return Q

def awq(W, bits, H, mag, grid=np.linspace(0, 1, 11)):
    """AWQ (Lin et al., 2024): scale input channel j of the weights up by
    s_j = mean|x_j|^alpha and its inputs down by the same factor, so the
    channels that matter most get finer steps; alpha chosen on calibration
    data. The scaling is exact in full precision."""
    best = None
    for a in grid:
        s = mag ** a; s /= np.sqrt(s.max() * s.min())
        Q = rtn(W * s, bits, "group 128") / s
        D = W - Q
        e = np.sum((D @ H) * D)                                   # calibration output error, from H
        if best is None or e < best[0]:
            best = (e, a, Q)
    return best[2], best[1]

def awq_clip(W, bits, H, mag, alpha, n_grid=20, max_shrink=0.5):
    """The second half of AWQ: after the scaling, for every group of 128
    weights in a row, search how far to clip its range before rounding
    (shrinking it by up to half), keeping whichever gives the least output
    error, here measured with the group's block of H."""
    s = mag ** alpha; s /= np.sqrt(s.max() * s.min())
    Ws = W * s
    Hs = H / s[:, None] / s[None, :]                               # H for the scaled inputs
    Q = np.zeros_like(Ws)
    for g in range(0, Ws.shape[1], 128):
        w, h = Ws[:, g:g + 128], Hs[g:g + 128, g:g + 128]
        top = np.abs(w).max(1, keepdims=True)
        best_err, best_q = np.full(w.shape[0], np.inf), np.zeros_like(w)
        for i in range(n_grid):
            t = top * (1 - max_shrink * i / n_grid)
            q = quantize(np.clip(w, -t, t), bits, axis=1)
            d = q - w
            e = np.sum((d @ h) * d, axis=1)
            better = e < best_err
            best_err[better], best_q[better] = e[better], q[better]
        Q[:, g:g + 128] = best_q
    return Q / s

# Worked: 8 weights to 4 bits.
w = np.array([-0.31, -0.12, 0.0, 0.04, 0.09, 0.18, 0.22, 0.47])
s, z = (w.max() - w.min()) / 15, np.round(-w.min() / ((w.max() - w.min()) / 15))
print(f"worked, 8 weights to 4 bits: scale {s:.4f}, zero-point {z:.0f}")
print(f"  integers {np.clip(np.round(w / s) + z, 0, 15).astype(int).tolist()}")
print(f"  back     {np.round(quantize(w, 4), 3).tolist()}")


for L in (12, 22):
    X, W = np.load(f"X_{L}.npy").astype(np.float64), np.load(f"W_{L}.npy").astype(np.float64)
    Xcal, Xtest = X[:2048], X[2048:]
    H_big, mag_big = np.load(f"H_big_{L}.npy"), np.load(f"mag_big_{L}.npy")
    ref = Xtest @ W.T
    err = lambda Q: np.linalg.norm(ref - Xtest @ Q.T) / np.linalg.norm(ref)
    print(f"\n=== layer {L}: input channels' mean |x|: median {np.median(mag_big):.3f}, largest "
          f"{mag_big.max():.2f} ({mag_big.max() / np.median(mag_big):.0f}x); top 1% of channels carry "
          f"{np.sort(mag_big)[-49:].sum() / mag_big.sum():.0%} of the total")
    print("round-to-nearest, output error (held-out tokens):")
    print(f"{'':12s}" + "".join(f"{b:>8d} bits" for b in (8, 4, 3)))
    for g in ("per tensor", "per channel", "group 128"):
        print(f"{g:12s}" + "".join(f"{err(rtn(W, b, g)):13.2%}" for b in (8, 4, 3)))
    G64 = W.reshape(W.shape[0], -1, 64)
    print(f"4 bits, blocks of 64: asymmetric {err(quantize(G64, 4, axis=2).reshape(W.shape)):.2%}, "
          f"symmetric {err(symmetric(G64, 4, 2).reshape(W.shape)):.2%}, NF4 {err(nf4(G64, 2).reshape(W.shape)):.2%}")
    print("group 128, output error (held-out tokens):")
    H_small = 2 * Xcal.T @ Xcal
    for b in (4, 3):
        Qa, a = awq(W, b, H_big, mag_big)
        print(f"  {b} bits: round-to-nearest {err(rtn(W, b, 'group 128')):.2%}; GPTQ {err(gptq(W, b, H_big)):.2%}; "
              f"AWQ scaling only {err(Qa):.2%} (alpha {a:.1f}), with clipping {err(awq_clip(W, b, H_big, mag_big, a)):.2%}")
    print("  GPTQ at 4 bits with 2,048 calibration tokens, by damping (fraction of H's mean diagonal): " +
          ", ".join(f"{d:g}: {err(gptq(W, 4, H_small, damp=d)):.2%}" for d in (0.01, 0.1, 1.0)) +
          f"; on its own calibration tokens (0.01): "
          f"{np.linalg.norm(Xcal @ W.T - Xcal @ gptq(W, 4, H_small).T) / np.linalg.norm(Xcal @ W.T):.2%}")

The post printed:

```text
worked, 8 weights to 4 bits: scale 0.0520, zero-point 6
  integers [0, 4, 6, 7, 8, 9, 10, 15]
  back     [-0.312, -0.104, 0.0, 0.052, 0.104, 0.156, 0.208, 0.468]

=== layer 12: input channels' mean |x|: median 0.067, largest 0.52 (8x); top 1% of channels carry 3% of the total
round-to-nearest, output error (held-out tokens):
                   8 bits       4 bits       3 bits
per tensor          4.95%       72.79%       94.82%
per channel         1.08%       18.17%       38.32%
group 128           0.64%       10.98%       23.31%
4 bits, blocks of 64: asymmetric 9.66%, symmetric 11.74%, NF4 9.34%
group 128, output error (held-out tokens):
  4 bits: round-to-nearest 10.98%; GPTQ 8.99%; AWQ scaling only 10.90% (alpha 0.2), with clipping 10.37%
  3 bits: round-to-nearest 23.31%; GPTQ 19.44%; AWQ scaling only 23.25% (alpha 0.2), with clipping 20.52%
  GPTQ at 4 bits with 2,048 calibration tokens, by damping (fraction of H's mean diagonal): 0.01: 12.78%, 0.1: 10.48%, 1: 9.46%; on its own calibration tokens (0.01): 3.76%

=== layer 22: input channels' mean |x|: median 0.146, largest 8.53 (58x); top 1% of channels carry 12% of the total
round-to-nearest, output error (held-out tokens):
                   8 bits       4 bits       3 bits
per tensor          3.69%       57.74%       85.07%
per channel         1.03%       17.38%       36.14%
group 128           0.63%       10.64%       22.62%
4 bits, blocks of 64: asymmetric 9.40%, symmetric 11.53%, NF4 8.51%
group 128, output error (held-out tokens):
  4 bits: round-to-nearest 10.64%; GPTQ 6.87%; AWQ scaling only 8.15% (alpha 0.4), with clipping 7.70%
  3 bits: round-to-nearest 22.62%; GPTQ 15.06%; AWQ scaling only 17.13% (alpha 0.4), with clipping 15.47%
  GPTQ at 4 bits with 2,048 calibration tokens, by damping (fraction of H's mean diagonal): 0.01: 8.23%, 0.1: 7.03%, 1: 6.95%; on its own calibration tokens (0.01): 2.64%
```

## What it buys at decode

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/quantization-from-scratch/#what-it-buys-at-decode)

In [ ]:
# What 4-bit weights buy at decode, by batch size, for an 8B model on an
# H100 (3.35 TB/s; 989 TFLOP/s dense BF16). One decode step for a batch of
# B sequences reads every weight once and does 2 FLOPs per weight per
# sequence; its time is at least the larger of the two ("How a GPU Runs Deep Learning"'s roofline;
# KV-cache reads and dequantization ignored). INT4 weights with group
# scales take about 4.25 bits each.
N, BW, FLOPS = 8e9, 3.35e12, 989e12
print(f"{'batch':>6s} {'BF16 step':>10s} {'INT4 step':>10s} {'speedup':>8s}   bound by")
for B in (1, 8, 64, 128, 256, 1024):
    compute = 2 * N * B / FLOPS
    bf16, int4 = max(2 * N / BW, compute), max(4.25 / 8 * N / BW, compute)
    print(f"{B:6d} {bf16 * 1e3:8.2f} ms {int4 * 1e3:8.2f} ms {bf16 / int4:7.1f}x   "
          f"BF16 {'memory' if 2 * N / BW > compute else 'compute'}, INT4 {'memory' if 4.25 / 8 * N / BW > compute else 'compute'}")
print(f"compute-bound from a batch of {4.25 / 8 * N / BW * FLOPS / (2 * N):.0f} (INT4) and "
      f"{2 * N / BW * FLOPS / (2 * N):.0f} (BF16)")

The post printed:

```text
 batch  BF16 step  INT4 step  speedup   bound by
     1     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
     8     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
    64     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
   128     4.78 ms     2.07 ms     2.3x   BF16 memory, INT4 compute
   256     4.78 ms     4.14 ms     1.2x   BF16 memory, INT4 compute
  1024    16.57 ms    16.57 ms     1.0x   BF16 compute, INT4 compute
compute-bound from a batch of 78 (INT4) and 295 (BF16)
```